# Train Models

This notebook trains the two models for the Quora Question Pairs deliverable:

1. **Baseline** — TF-IDF on the concatenated questions + Logistic Regression.
2. **Improved** — engineered feature vector (Jaccard, length, edit distance, TF-IDF cosine, …) + XGBoost.

All functions live in `utils.py`; this notebook only orchestrates the pipeline and saves the trained artefacts to `./models/`.

**Reproducibility.** The random seed is fixed (`utils.RANDOM_STATE = 123`) and the train/val/test split is the exact one specified in the deliverable PDF, so every group obtains identical partitions.

**Data location.** The notebook reads from `$HOME/Datasets/QuoraQuestionPairs/quora_data.csv` as required.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier

import utils

MODELS_DIR = './models'
os.makedirs(MODELS_DIR, exist_ok=True)

# As required by the deliverable: do not overwrite previously trained models.
BASELINE_PATH = os.path.join(MODELS_DIR, 'baseline.joblib')
IMPROVED_PATH = os.path.join(MODELS_DIR, 'improved.joblib')

print('utils loaded from', utils.__file__)
print('models will be saved to', os.path.abspath(MODELS_DIR))

utils loaded from c:\Users\Miki\OneDrive - Universitat de Barcelona\MÀSTER\2nS\NLP\REPO\Natural-Language-Processing\DELIVERABLE_1\Miki's-things\utils.py
models will be saved to c:\Users\Miki\OneDrive - Universitat de Barcelona\MÀSTER\2nS\NLP\REPO\Natural-Language-Processing\DELIVERABLE_1\Miki's-things\models


## 1. Load data and reproduce the official split

In [2]:
df = utils.load_quora_dataframe()
print('full dataset:', df.shape)
df.head()

full dataset: (323432, 6)


,id,qid1,qid2,question1,question2,is_duplicate
0,346692,38482,10706,Why do I get easily bored with everything?,Why do I get bored with things so quickly and ...,1
1,327668,454117,345117,How do I study for Honeywell company recruitment?,How do I study for Honeywell company recruitme...,1
2,272993,391373,391374,Which search engine algorithm is Quora using?,Why is Quora not using reliable search engine?,0
3,54070,82673,95496,How can I smartly cut myself?,Can someone who thinks about suicide for 7 yea...,0
4,46450,38384,72436,How do I see who is viewing my Instagram videos?,Can one tell who viewed my Instagram videos?,1


In [3]:
train_df, val_df, test_df = utils.split_quora(df)
print('train_df.shape =', train_df.shape)
print('val_df.shape   =', val_df.shape)
print('test_df.shape  =', test_df.shape)
print('class balance (train) =', train_df['is_duplicate'].mean().round(4))

train_df.shape = (291897, 6)
val_df.shape   = (15363, 6)
test_df.shape  = (16172, 6)
class balance (train) = 0.3686


## 2. Baseline model — TF-IDF + Logistic Regression

Following the slide *"Baseline pipeline: TF-IDF + Logistic Regression"* from lecture 3 we concatenate the two questions with a `[SEP]` token and feed the resulting string to a TF-IDF vectoriser plus a logistic-regression classifier.

**Why a baseline first?** It gives us a number to beat and exposes the limitations we then try to fix with engineered features:

* TF-IDF on the *concatenation* cannot see the *interaction* between the two questions — it just measures the union vocabulary.
* It is bag-of-words: word order is lost.
* It cannot recognise a paraphrase that uses different vocabulary (e.g. *"automobile"* vs *"car"*).
* It struggles on short questions where the [SEP] dilutes the signal.

In [4]:
if os.path.exists(BASELINE_PATH):
    print(f'{BASELINE_PATH} already exists — skipping training (as required).')
    baseline = joblib.load(BASELINE_PATH)
else:
    vec, clf = utils.fit_baseline(train_df, max_features=20000)
    baseline = {'vec': vec, 'clf': clf}
    joblib.dump(baseline, BASELINE_PATH)
    print(f'Saved baseline to {BASELINE_PATH}')

Saved baseline to ./models\baseline.joblib


In [5]:
# Quick look at the baseline metrics (the proper evaluation lives in reproduce_results.ipynb)
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    proba, _ = utils.predict_baseline(baseline['vec'], baseline['clf'], d)
    print(f'baseline {name}:', utils.evaluate(d['is_duplicate'].values, proba))

baseline train: {'roc_auc': np.float64(0.8704707813769481), 'precision': 0.7735033134898349, 'recall': 0.6400152417773399}
baseline val: {'roc_auc': np.float64(0.8422432635005163), 'precision': 0.7368871761935346, 'recall': 0.6073760367037233}
baseline test: {'roc_auc': np.float64(0.8473187249733485), 'precision': 0.7528066952439273, 'recall': 0.6139503912102547}


## 3. Improved model — engineered features + XGBoost

The improvement is to compute, *for each pair of questions*, a small feature vector that captures the **interaction** between the two questions.  Each member of the team owns one block of these features (see `main.pdf` for the exact division):

| Block | Owner | Features |
|------|------|---------|
| Lexical / set-based | Member A | jaccard, jaccard_content, word_overlap_ratio, common_words, length features (token + char) |
| Edit distance | Member B | char_edit, char_edit_norm, tok_edit, tok_edit_norm, common_prefix |
| TF-IDF cosine from scratch | Member C | tfidf_cosine_scratch (+ sklearn cross-check) |

All from-scratch implementations live in `utils.py`.

We feed this feature vector to **XGBoost**, a gradient-boosted-trees classifier, which is well-suited to small dense numerical feature vectors with non-linear interactions.

### 3.1 Fit the two TF-IDF vectorisers
We fit two parallel TF-IDF vectorisers — Member C's hand-coded one and the sklearn reference — on the *training* set only, to avoid leakage.

In [6]:
all_train_questions = pd.concat([train_df['question1'], train_df['question2']]).tolist()

tfidf_scratch = utils.TfidfFromScratch(min_df=2, sublinear_tf=True).fit(all_train_questions)
tfidf_sklearn = TfidfVectorizer(min_df=2, sublinear_tf=True, ngram_range=(1, 1))
tfidf_sklearn.fit(all_train_questions)

print('vocab size (scratch) :', len(tfidf_scratch.vocabulary_))
print('vocab size (sklearn) :', len(tfidf_sklearn.vocabulary_))

vocab size (scratch) : 45553
vocab size (sklearn) : 44867


### 3.2 Build the feature matrices
This is the slow step on the full Quora dataset (~5–10 min on a laptop).

In [ ]:
print('Building train features...')
X_train = utils.build_feature_matrix(train_df, tfidf_scratch, tfidf_sklearn)
y_train = train_df['is_duplicate'].astype(int).values
print('Building val features...')
X_val   = utils.build_feature_matrix(val_df,   tfidf_scratch, tfidf_sklearn)
y_val   = val_df['is_duplicate'].astype(int).values
print('Building test features...')
X_test  = utils.build_feature_matrix(test_df,  tfidf_scratch, tfidf_sklearn)
y_test  = test_df['is_duplicate'].astype(int).values

print('shapes:', X_train.shape, X_val.shape, X_test.shape)
pd.DataFrame(X_train[:5], columns=utils.FEATURE_NAMES)

Building train features...


### 3.3 Train XGBoost

In [ ]:
if os.path.exists(IMPROVED_PATH):
    print(f'{IMPROVED_PATH} already exists — skipping training (as required).')
    improved = joblib.load(IMPROVED_PATH)
else:
    xgb = XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric='logloss',
        random_state=utils.RANDOM_STATE,
        n_jobs=-1,
        tree_method='hist',
    )
    xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    improved = {
        'tfidf_scratch':  tfidf_scratch,
        'tfidf_sklearn':  tfidf_sklearn,
        'model':          xgb,
        'feature_names':  utils.FEATURE_NAMES,
    }
    joblib.dump(improved, IMPROVED_PATH)
    print(f'Saved improved model to {IMPROVED_PATH}')

In [ ]:
# Sanity check on the training set
for name, X, y in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    p = improved['model'].predict_proba(X)[:, 1]
    print(f'improved {name}:', utils.evaluate(y, p))

### 3.4 Feature importance (informational)

In [ ]:
imp = pd.Series(improved['model'].feature_importances_, index=utils.FEATURE_NAMES)
imp.sort_values(ascending=False)

Done.  Run `reproduce_results.ipynb` to load the saved models from disk and produce the official metrics dataframe.